# Sesión 23 · Plotly I: gráficos interactivos y visualización dinámica
**Visualización I · Universidad Externado de Colombia**  
**Docente: Adriana Pineda** · Notebook maestro para Google Colab

Trabajaremos con `llamadas123_2025.xlsx`, `llamadas123.xlsx` y `localidades.csv`. La unidad inicial es una **fila registrada**: no asumiremos que cada fila corresponde a un incidente único.

## Ruta de aprendizaje
1. Introducción a Plotly y entornos web/Jupyter.
2. Carga, diagnóstico y preparación de datos.
3. Zoom, hover, selección y filtrado dinámico.
4. Conversión de Seaborn a Plotly.
5. Barras, líneas, dispersión, histogramas y cajas.
6. Personalización orientada a la audiencia.
7. Exploración dinámica con controles básicos.
8. Reflexión crítica y conexión con Power BI.
9. Aplicación autónoma.

**Resultados:** convertir un gráfico estático en su equivalente interactivo, distinguir selección de filtrado y justificar cuándo la interacción ayuda a responder una pregunta analítica. Esta sesión trabaja gráficos individuales; la coordinación de un dashboard corresponde a la sesión 24.

**Cómo usarlo:** sube este `.ipynb` a Colab, guarda una copia en Drive y ejecuta las celdas en orden. En la sección 2 se solicitarán las tres bases si todavía no están en el entorno. Los archivos con sufijos como `(3)` también se reconocen. El entorno de Colab se borra al finalizar la sesión: vuelve a subir los datos cuando sea necesario.

## 1. ¿Qué aporta Plotly?
Matplotlib permite construir y controlar figuras; Seaborn simplifica gráficos estadísticos y sus agrupaciones. Plotly genera figuras que un navegador representa con JavaScript: en Colab/Jupyter podemos explorar datos con el cursor y controles dentro del propio gráfico.

| Aspecto | Seaborn | Plotly |
|---|---|---|
| Sintaxis básica | `sns.scatterplot(data=..., x=..., y=..., hue=...)` | `px.scatter(data_frame=..., x=..., y=..., color=...)` |
| Resultado habitual | Figura estática de Matplotlib | Figura interactiva |
| Diseño | Ejes y objetos de Matplotlib | Trazas (series de datos) y layout (diseño) |
| Exploración | Se modifica el código para otra vista | Zoom, hover, selección y controles |
| Uso | Informe fijo y exploración estadística | Exploración por una audiencia analítica |

**Plotly Express (`px`)** crea figuras con pocas instrucciones. **Graph Objects (`go`)** permite controlar trazas y menús. Express devuelve una figura que también puede personalizarse con `update_layout` y `update_traces`.

La interactividad no corrige una mala pregunta ni un denominador inadecuado. Primero definimos audiencia, pregunta y unidad de análisis.

In [ ]:
# Instalar solo dependencias que falten. En Colab suelen estar disponibles.
import importlib.util, subprocess, sys
paquetes = {'pandas':'pandas', 'numpy':'numpy', 'openpyxl':'openpyxl',
            'matplotlib':'matplotlib', 'seaborn':'seaborn', 'plotly':'plotly'}
faltantes = [p for m,p in paquetes.items() if importlib.util.find_spec(m) is None]
if faltantes:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *faltantes])
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from pathlib import Path
from IPython.display import display
import re, unicodedata
try:
    import google.colab
    EN_COLAB = True
except ImportError:
    EN_COLAB = False
pio.renderers.default = 'colab' if EN_COLAB else 'plotly_mimetype'
sns.set_theme(style='whitegrid')
px.defaults.template = 'plotly_white'
CONFIG = {'displaylogo': False, 'scrollZoom': True,
          'modeBarButtonsToAdd': ['select2d', 'lasso2d']}
def mostrar(fig):
    fig.show(config=CONFIG)
print('Entorno listo. Plotly:', __import__('plotly').__version__)

## 2. Preparación del entorno y de las bases
### 2.1 Cargar los archivos originales
No se sustituyen los datos por una simulación. La búsqueda reconoce el nombre base y sufijos de descarga. Si hay varias copias, exige escoger una para evitar utilizar una versión equivocada. En Jupyter local, cambia `CARPETA` a la carpeta donde guardaste las bases.

In [ ]:
CARPETA = Path('/content') if EN_COLAB else Path('.')
def buscar(nombre, extension):
    patron = re.compile(re.escape(nombre) + r'(?:\(\d+\))?' + re.escape(extension), re.I)
    candidatos = sorted(p for p in CARPETA.iterdir() if p.is_file() and patron.fullmatch(p.name))
    exacto = CARPETA / (nombre + extension)
    if exacto in candidatos:
        return exacto
    if len(candidatos) > 1:
        raise ValueError(f'Hay varias copias de {nombre}: {[p.name for p in candidatos]}. Conserva una o usa el nombre exacto.')
    return candidatos[0] if candidatos else None
requeridos = [('llamadas123_2025','.xlsx'), ('llamadas123','.xlsx'), ('localidades','.csv')]
if any(buscar(*r) is None for r in requeridos) and EN_COLAB:
    from google.colab import files
    print('Selecciona las tres bases desde tu computador.')
    files.upload()
rutas = {nom: buscar(nom, ext) for nom, ext in requeridos}
if any(p is None for p in rutas.values()):
    raise FileNotFoundError('Faltan archivos: ' + ', '.join(n for n,p in rutas.items() if p is None))
originales = {'2025': pd.read_excel(rutas['llamadas123_2025']),
              '2026': pd.read_excel(rutas['llamadas123'])}
# Mantener skipinitialspace=True evita espacios después de los separadores.
localidades_original = pd.read_csv(rutas['localidades'], skipinitialspace=True)
for nombre, tabla in [*originales.items(), ('localidades', localidades_original)]:
    print(nombre, tabla.shape, tabla.columns.tolist())
    display(tabla.head(3))

### 2.2 Diagnóstico antes de transformar
Revisamos faltantes, duplicados de identificador, fechas y unidades. **No eliminaremos filas por tener un identificador repetido:** su significado debe comprobarse antes de modificar la unidad de análisis.

Pregunta: ¿es correcto llamar “número de incidentes” a un conteo de filas sin verificar la unicidad de `NUMERO_INCIDENTE`?

In [ ]:
for fuente, d in originales.items():
    print('BASE', fuente)
    display(d.isna().sum().rename('faltantes').to_frame())
    print('Identificadores repetidos:', d['NUMERO_INCIDENTE'].duplicated().sum())
    print('Unidades de edad:')
    display(d['UNIDAD'].value_counts(dropna=False).to_frame('filas'))

### 2.3 Transformaciones mínimas y verificables
- Se conservan las bases originales y una columna que identifica la fuente.
- Se normaliza el **código** de localidad para unir tablas; no se unen nombres con tildes o variantes.
- Se conservan códigos como `SIN_D` y `FUERA_DE_BOGOTA`.
- Se valida una unión de muchas filas a una localidad; no deben multiplicarse las filas.
- Para los gráficos de edad se incluyen únicamente registros cuya unidad corresponde a **años**, con edades entre 0 y 110. Meses, días y horas no se interpretan como años. Es una regla explícita del ejercicio, no una limpieza universal.
- Se emplea la fecha de inicio del desplazamiento del móvil; no se interpreta como fecha de recepción.

Los períodos disponibles se calculan a partir de los archivos. Los totales de 2025 y 2026 no representan ventanas equivalentes; tampoco miden tasas poblacionales ni tiempos de respuesta.

**Limpieza para una gráfica legible:** se reducen espacios repetidos, se unifica la escritura de las prioridades y tipos, y se utiliza el nombre oficial de la localidad desde la tabla de referencia. Los vacíos se rotulan como “Sin información”. No se agrupan tipos distintos por semejanza de texto sin verificar su significado. Los gráficos de ranking usan barras horizontales para que las etiquetas largas sean legibles; los ejes siempre indican la unidad. Un gráfico limpio también requiere categorías coherentes y una agregación correcta, además de formato visual.

In [ ]:
def texto(x):
    if pd.isna(x): return ''
    return re.sub(r'\s+', ' ', str(x)).strip()
def clave(x):
    s = texto(x)
    return str(int(float(s))) if re.fullmatch(r'\d+(?:\.0+)?', s) else s.upper()
def normalizar(x):
    return ''.join(c for c in unicodedata.normalize('NFKD', texto(x).upper()) if not unicodedata.combining(c))
localidades = localidades_original.copy()
localidades.columns = localidades.columns.str.strip()
localidades['codigo'] = localidades['CODIGO_LOCALIDAD'].map(clave)
if localidades['codigo'].duplicated().any():
    raise ValueError('La tabla de localidades tiene códigos repetidos: revisa antes de unir.')
localidades = localidades.rename(columns={'LOCALIDAD':'localidad', 'ZONA':'zona', 'SUBRED_SALUD':'subred'})
for c in ['localidad','zona','subred']:
    if c not in localidades: localidades[c] = 'Sin información'
    localidades[c] = localidades[c].map(lambda x: texto(x) or 'Sin información')
preparadas = []
for fuente, original in originales.items():
    d = original.copy()
    d.columns = d.columns.str.strip()
    d['fuente'] = fuente
    d['codigo'] = d['CODIGO_LOCALIDAD'].map(clave)
    d['fecha'] = pd.to_datetime(d['FECHA_INICIO_DESPLAZAMIENTO_MOVIL'], errors='coerce')
    d['prioridad'] = d['PRIORIDAD_FINAL'].map(lambda x: normalizar(x).title() or 'Sin información')
    d['tipo'] = d['TIPO_INCIDENTE'].map(lambda x: texto(x).upper() or 'Sin información')
    unidad = d['UNIDAD'].map(normalizar)
    es_anios = unidad.isin(['AÑOS','ANOS','A–OS','A¤OS'])
    edad = pd.to_numeric(d['EDAD'], errors='coerce')
    d['edad_anios'] = edad.where(es_anios & edad.between(0,110))
    preparadas.append(d)
datos = pd.concat(preparadas, ignore_index=True)
n_antes = len(datos)
datos = datos.merge(localidades[['codigo','localidad','zona','subred']], on='codigo', how='left', validate='many_to_one', indicator=True)
print('Filas sin coincidencia de localidad:', (datos['_merge'] == 'left_only').sum())
for c in ['localidad','zona','subred']:
    datos[c] = datos[c].fillna('Sin información')
assert len(datos) == n_antes
datos['dia'] = datos['fecha'].dt.normalize()
datos['mes'] = datos['fecha'].dt.strftime('%Y-%m')
cobertura = datos.groupby('fuente').agg(filas=('fuente','size'), inicio=('fecha','min'), fin=('fecha','max'), edades_en_anios=('edad_anios','count'))
display(cobertura)
print('Fechas faltantes:', datos['fecha'].isna().sum())
print('Prioridades después de limpiar:')
display(datos['prioridad'].value_counts(dropna=False).to_frame('registros'))
print('Ejemplos de etiquetas originales y limpias:')
display(datos[['PRIORIDAD_FINAL','prioridad','TIPO_INCIDENTE','tipo','LOCALIDAD','localidad']].head(8))
# Período principal: no se mezclan las ventanas de 2025 y 2026.
base = datos.loc[datos['fuente'].eq('2026')].copy()
validos = base.dropna(subset=['fecha'])
diario = validos.groupby('dia').size().rename('registros').reset_index()
barras = base.groupby('localidad').size().rename('registros').reset_index().sort_values('registros',ascending=False)
# Cada punto representa una localidad: volumen y edad media en registros con edad utilizable.
resumen = base.groupby(['localidad','zona']).agg(registros=('fuente','size'), edad_media=('edad_anios','mean'), n_edad=('edad_anios','count')).reset_index()
resumen = resumen.dropna(subset=['edad_media'])
display(resumen.head())

## 3. Fundamentos de interactividad
### 3.1 Zoom: inspeccionar una región sin cambiar los datos
**Pregunta:** ¿qué variaciones se observan dentro de un intervalo corto de fechas?

En la barra superior elige la lupa y arrastra un rectángulo. Prueba los botones `+`, `−`, desplazamiento y restablecer ejes; Reset axes permite recuperar la vista. El deslizador inferior controla la ventana visible. El zoom cambia los límites del eje, **no recalcula los datos**.

**Ejercicio guiado:** acerca una semana con alta variación, pasa el cursor por su máximo y vuelve a la vista completa. ¿La impresión de variabilidad cambia al ampliar la escala? Un salto también puede reflejar cobertura incompleta.

In [ ]:
fig_zoom = px.line(diario, x='dia', y='registros', markers=True,
                   title='Registros por día · fuente 2026', labels={'dia':'Fecha de desplazamiento','registros':'Número de registros'})
fig_zoom.update_xaxes(rangeslider_visible=True)
fig_zoom.update_layout(hovermode='x unified', height=470)
mostrar(fig_zoom)

### 3.2 Hover: información adicional al pasar el cursor
**Pregunta:** ¿cuántos registros sustentan la edad promedio de cada localidad?

`hover_data` incorpora variables adicionales sin saturar el gráfico. Una media puede basarse en pocas edades: siempre mostramos `n_edad` para valorar su cobertura. No se muestran identificadores individuales.

Nota: n_edad es el número de registros con una edad válida en años en cada localidad. Esos registros se utilizan para calcular la edad promedio.
Por ejemplo, si una localidad tiene:
- 1.000 registros en total (registros).
- 650 registros con edad válida en años (n_edad).
La edad media se calcula con esas 650 edades, porque los demás registros tienen la edad ausente, expresada en otras unidades o fuera del rango definido para el ejercicio.
Mostrar n_edad permite saber cuántas edades sustentan el promedio. No es la edad ni el número total de registros.

**Ejercicio guiado:** compara dos localidades con edades medias parecidas. ¿También tienen tamaños y cobertura de edad semejantes? La asociación entre estos resúmenes no demuestra causalidad ni describe necesariamente a toda la población.

In [ ]:
fig_hover = px.scatter(resumen, x='registros', y='edad_media', color='zona',
    hover_name='localidad', hover_data={'n_edad':True, 'edad_media':':.1f', 'registros':':,d'},
    labels={'registros':'Número de registros','edad_media':'Edad media (años)','zona':'Zona','n_edad':'Registros con edad'},
    title='Volumen y edad media por localidad · fuente 2026')
fig_hover.update_traces(marker={'size':12,'opacity':0.85})
mostrar(fig_hover)

### 3.3 Selección: destacar un subconjunto de puntos
El modo de selección rectangular y el lazo (herramienta para seleccionar puntos dibujando una figura libre alrededor de ellos, como si los rodearas con un cordón) destacan puntos; los demás permanecen atenuados. Cada punto aquí es una localidad, no una persona.

**Ejercicio guiado:** arrastra un rectángulo alrededor de localidades con mayor volumen. Cambia a lazo en la barra y selecciona un grupo irregular. Pasa el cursor por sus puntos o por el contorno más cercano a los puntos para identificar localidades. Limpia la selección con doble clic.

**Pregunta:** ¿las localidades seleccionadas pertenecen a una sola zona? ¿Qué criterio empleaste para elegirlas?

En una figura normal de Plotly, esta selección ocurre en el navegador: **no crea automáticamente una tabla filtrada en Python**, ni coordina otros gráficos. No requiere callbacks ni `FigureWidget` para este ejercicio.

In [ ]:
fig_seleccion = go.Figure(fig_hover)
fig_seleccion.update_layout(title='Selecciona localidades por volumen y edad media', dragmode='select')
fig_seleccion.update_traces(selected={'marker':{'opacity':1}}, unselected={'marker':{'opacity':0.2}})
mostrar(fig_seleccion)

In [ ]:
fig_seleccion.update_layout(
    dragmode="lasso",
    hovermode="closest",
    hoverdistance=30
)

mostrar(fig_seleccion)

### 3.4 Filtrado dinámico: cambiar el subconjunto representado
**Pregunta:** ¿cambia la distribución de registros por localidad al elegir una prioridad?

El menú modifica los valores y categorías representados a partir de subconjuntos precomputados. Incluye `Todas` para recuperar la población de filas de la fuente 2026. El código conserva la misma lista ordenada de localidades: facilita comparar y muestra ceros cuando una localidad no tiene registros en esa prioridad.

**Ejercicio guiado:** selecciona `Alta`, luego otra prioridad y regresa a `Todas`. Observa las barras y sus valores emergentes. ¿Se mantiene la localidad con mayor número de registros?

**Diferencia:** seleccionar resalta puntos de la vista actual; filtrar redefine qué registros alimentan la representación. El menú no altera el DataFrame original. El eje vertical se reajusta: al comparar filtros, lee los valores y la escala.

In [ ]:
orden = barras['localidad'].tolist()
opciones = ['Todas'] + sorted(base['prioridad'].unique().tolist())
conteos = {}
for opcion in opciones:
    sub = base if opcion == 'Todas' else base.loc[base['prioridad'].eq(opcion)]
    conteos[opcion] = sub.groupby('localidad').size().reindex(orden, fill_value=0)
fig_filtro = go.Figure(go.Bar(x=orden, y=conteos['Todas'].tolist(), marker_color='#287C8E',
    hovertemplate='%{x}<br>Registros: %{y:,d}<extra></extra>'))
botones = [dict(label=o, method='update', args=[{'y':[conteos[o].tolist()]},
            {'title':f'Registros por localidad · prioridad: {o} · fuente 2026'}]) for o in opciones]
fig_filtro.update_layout(title='Registros por localidad · prioridad: Todas · fuente 2026',
    xaxis_title='Localidad', yaxis_title='Número de registros', height=540,
    margin=dict(t=130,b=150), updatemenus=[dict(buttons=botones, x=0, y=1.2, xanchor='left', yanchor='top')])
fig_filtro.update_xaxes(tickangle=-55)
mostrar(fig_filtro)

In [ ]:
# Limpiar los datos

# Copia para conservar los datos del gráfico inicial
base_limpia = base.copy()

# Normalizar espacios, mayúsculas y códigos como 1.0
prioridad_original = base_limpia["PRIORIDAD_FINAL"]

prioridad_normalizada = (
    prioridad_original
    .astype("string")
    .str.strip()
    .str.upper()
    .str.replace(r"\s+", " ", regex=True)
    .str.replace(r"^([1-4])\.0+$", r"\1", regex=True)
)

# Unificar códigos y etiquetas que ya estaban escritas
mapa_prioridad = {
    "1": "Crítica",
    "2": "Alta",
    "3": "Media",
    "4": "Baja",
    "CRITICA": "Crítica",
    "CRÍTICA": "Crítica",
    "ALTA": "Alta",
    "MEDIA": "Media",
    "BAJA": "Baja"
}

base_limpia["prioridad"] = prioridad_normalizada.map(mapa_prioridad)

# Revisar valores no reconocidos antes de continuar
no_reconocidos = (
    prioridad_normalizada.notna()
    & prioridad_normalizada.ne("")
    & base_limpia["prioridad"].isna()
)

if no_reconocidos.any():
    print("Valores no reconocidos: revisar su significado.")
    display(
        prioridad_original[no_reconocidos]
        .value_counts()
        .to_frame("Registros")
    )

base_limpia["prioridad"] = (
    base_limpia["prioridad"].fillna("Sin información")
)

# Comparación antes y después de limpiar
display(
    base_limpia.groupby(
        ["PRIORIDAD_FINAL", "prioridad"],
        dropna=False
    ).size().reset_index(name="Registros")
)

## 4. De Seaborn a Plotly: misma pregunta, mismos datos
**Pregunta:** ¿cuáles son las diez localidades con más registros en la fuente 2026?

Usamos la misma tabla ya agregada y el mismo orden en ambas bibliotecas. Seaborn no vuelve a estimar medias de los conteos. Plotly tampoco realiza una agregación implícita.

Primero observa la comparación estática; luego explora los valores exactos con hover y prueba el zoom. **La diferencia no tiene que estar en la forma de las barras:** está en cómo puede explorarlas el lector.

El top 10 omite otras localidades: sirve para una pregunta de ranking, no para presentar toda la distribución.

In [ ]:
top10 = barras.head(10).sort_values('registros')
fig, ax = plt.subplots(figsize=(10,5))
sns.barplot(data=top10, x='registros', y='localidad', order=top10['localidad'].tolist(),
            color='#287C8E', errorbar=None, ax=ax)
ax.set(title='Diez localidades con más registros · fuente 2026', xlabel='Número de registros', ylabel='Localidad')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
fig_barras = px.bar(top10, x='registros', y='localidad', orientation='h',
    title='Diez localidades con más registros · fuente 2026',
    labels={'registros':'Número de registros','localidad':'Localidad'}, color_discrete_sequence=['#287C8E'])
fig_barras.update_yaxes(categoryorder='array', categoryarray=top10['localidad'].tolist())
fig_barras.update_layout(height=500)
mostrar(fig_barras)

### Segunda conversión: dispersión
Ambas versiones utilizan `resumen`, con un punto por localidad. Seaborn usa `hue` para separar zonas; Plotly usa `color` y añade datos emergentes. Haz clic en una categoría de la leyenda de Plotly para ocultarla; doble clic aísla una serie. **Ocultar una traza no recalcula estadísticas de las otras**.

**Para discutir:** ¿qué descubrimiento fue más fácil con hover? ¿Cuándo bastaría la figura estática?

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
sns.scatterplot(data=resumen, x='registros', y='edad_media', hue='zona', s=90, ax=ax)
ax.set(title='Volumen y edad media por localidad · fuente 2026', xlabel='Número de registros', ylabel='Edad media (años)')
ax.legend(title='Zona', bbox_to_anchor=(1.02,1), loc='upper left')
plt.tight_layout(); plt.show()
mostrar(fig_hover)

## 5. Gráficos interactivos según la pregunta
Ya construimos **barras** (comparación), **líneas** (evolución temporal) y **dispersión** (relación entre resúmenes). Completamos con histogramas y cajas.

### 5.1 Histograma: distribución de las edades
La altura representa registros en intervalos de edad. `nbins` sugiere la cantidad de intervalos; no implica que cada barra corresponda a una edad exacta. Excluimos edades ausentes y unidades distintas de años; el título comunica esta selección. **Pregunta:** ¿en qué rangos se concentra la distribución? ¿Cambiar el número de intervalos modifica su apariencia?

In [ ]:
edades = base.dropna(subset=['edad_anios']).copy()
if edades.empty: raise ValueError('No hay edades válidas en años; revisa la columna UNIDAD.')
fig_hist = px.histogram(edades, x='edad_anios', nbins=25,
    title=f'Distribución de edades en años · fuente 2026 · n={len(edades):,}',
    labels={'edad_anios':'Edad (años)'}, color_discrete_sequence=['#287C8E'])
fig_hist.update_layout(yaxis_title='Número de registros', bargap=0.04)
mostrar(fig_hist)

### 5.2 Cajas: comparar distribución de edad según prioridad
La caja resume cuartiles y mediana; los bigotes de Plotly se basan en la regla de 1,5 veces el rango intercuartílico. Un valor fuera de los bigotes no demuestra un error. Para mantener fluidez se ocultan puntos individuales (`points=False`); no se eliminan del cálculo.

**Ejercicio:** usa hover para leer los resúmenes de cada grupo. ¿Qué prioridad tiene mayor mediana? ¿La mayor mediana implica necesariamente mayor dispersión? Revisa también el tamaño de los grupos.

In [ ]:
display(edades.groupby('prioridad')['edad_anios'].agg(['count','median','mean']).round(2))
fig_cajas = px.box(edades, x='prioridad', y='edad_anios', points=False,
    title='Edad por prioridad · registros con edad en años · fuente 2026',
    labels={'prioridad':'Prioridad','edad_anios':'Edad (años)'}, color_discrete_sequence=['#287C8E'])
fig_cajas.update_traces(quartilemethod='linear')
mostrar(fig_cajas)

## 6. Personalización: hacer útil la interacción
Una audiencia analítica necesita contexto, unidad, cobertura y valores precisos. El color debe tener una función; el hover debe añadir información relevante, no repetir columnas sin propósito.

Personalizamos el gráfico de ranking: título descriptivo, etiquetas con unidades, texto exterior y un hover breve. No formulamos un titular causal a partir de un conteo descriptivo.

**Ejercicio:** cambia solo el título para que responda a la pregunta elegida y explica si es descriptivo o declarativo. ¿Un lector puede entender el gráfico sin abrir el hover?

In [ ]:
fig_personalizado = go.Figure(fig_barras)
fig_personalizado.update_layout(title=dict(text='Mayor volumen de registros: diez localidades · fuente 2026', x=0.02),
    font=dict(family='Arial', size=13), margin=dict(l=140,r=80,t=90,b=60),
    xaxis=dict(title='Número de registros', rangemode='tozero'), showlegend=False)
fig_personalizado.update_traces(texttemplate='%{x:,d}', textposition='outside', cliponaxis=False,
    hovertemplate='<b>%{y}</b><br>Registros: %{x:,d}<extra></extra>')
mostrar(fig_personalizado)

## 7. Exploración dinámica: elegir fuente y localidad
Este segundo menú permite explorar series temporales por **combinación de fuente y localidad**, sin mezclar ventanas. Un solo control evita estados incoherentes entre dos menús independientes.

Se reindexan días entre el mínimo y el máximo de cada fuente. Un cero significa **ninguna fila registrada en el archivo para ese día y localidad**; no garantiza ausencia de incidentes en la ciudad. Primer y último día pueden tener cobertura parcial.

**Ejercicio:** selecciona una localidad en 2026 y después la misma en 2025. Observa las fechas antes de comparar. ¿Qué advertencia incluirías si un usuario intentara comparar sus totales?

El deslizador temporal modifica la ventana visible; el menú sí cambia el subconjunto representado. Son acciones diferentes.

In [ ]:
series = {}
for fuente in sorted(datos['fuente'].unique()):
    f = datos.loc[datos['fuente'].eq(fuente)].dropna(subset=['dia'])
    if f.empty: continue
    calendario = pd.date_range(f['dia'].min(), f['dia'].max(), freq='D')
    for loc in ['Todas'] + sorted(f['localidad'].unique()):
        sub = f if loc == 'Todas' else f.loc[f['localidad'].eq(loc)]
        series[f'{fuente} · {loc}'] = sub.groupby('dia').size().reindex(calendario, fill_value=0)
inicial = '2026 · Todas' if '2026 · Todas' in series else next(iter(series))
s = series[inicial]
fig_dinamico = go.Figure(go.Scatter(x=s.index, y=s.tolist(), mode='lines', line=dict(color='#287C8E'),
    hovertemplate='%{x|%d/%m/%Y}<br>Registros: %{y:,d}<extra></extra>'))
menu = [dict(label=k, method='update', args=[{'x':[v.index.tolist()], 'y':[v.tolist()]},
        {'title':f'Registros diarios · {k}', 'xaxis.autorange':True, 'yaxis.autorange':True}]) for k,v in series.items()]
fig_dinamico.update_layout(title=f'Registros diarios · {inicial}', height=530, margin=dict(t=135),
    xaxis_title='Fecha de desplazamiento', yaxis_title='Número de registros',
    updatemenus=[dict(buttons=menu, active=list(series).index(inicial), x=0,y=1.23, xanchor='left')])
fig_dinamico.update_xaxes(rangeslider_visible=True)
mostrar(fig_dinamico)

## 8. Reflexión crítica: ¿cuándo aporta valor?
Retomamos el diseño estratégico trabajado en Power BI: **audiencia → pregunta → evidencia → elección visual**. Agregar controles es una decisión comunicativa, no un objetivo por sí mismo.

| Situación | Decisión razonada |
|---|---|
| Un investigador necesita comparar prioridades | Un filtro facilita explorar la misma pregunta con subconjuntos |
| Un informe impreso debe mostrar un resultado central | Una figura estática con anotaciones puede comunicarlo mejor |
| Hay muchos puntos y datos adicionales relevantes | Hover evita recargar la figura; debe mantenerse el contexto visible |
| Se ofrecen controles que no cambian ninguna conclusión | Pueden introducir complejidad y distraer |
| Se usan filtros y el lector pierde el denominador | Mostrar fuente, período y subconjunto en el título |

**Discusión:**
1. ¿Qué información descubriste con una interacción que no era fácil leer en el gráfico estático?
2. ¿Cuándo el zoom puede exagerar una variación?
3. ¿Por qué selección, ocultamiento de leyenda y filtrado no son equivalentes?
4. ¿Qué decisión de color o etiqueta conservarías del bloque de Power BI?
5. ¿Qué interacción retirarías si el gráfico estuviera dirigido a alguien que solo necesita un hallazgo?
6. ¿Por qué estos conteos no permiten concluir qué localidad tiene mayor riesgo para un residente?

**Pistas docentes:** el zoom altera la escala visible; las tasas exigirían denominadores y cobertura comparables; los filtros deben conservar contexto. Una respuesta debe relacionar el control con una pregunta concreta.

## 9. Ejercicio de aplicación
**Audiencia sugerida:** equipo técnico que explora la composición de los registros de desplazamiento móvil en 2026.

Elige una pregunta sobre localidades, prioridad o tipo de incidente que pueda responderse con estas variables. Construye primero una figura estática y luego una figura Plotly equivalente.

Entrega en tu copia del Notebook:
1. Audiencia específica y pregunta analítica.
2. Unidad de análisis, fuente, período y tratamiento de faltantes.
3. Las dos versiones con iguales datos y agregación.
4. Una interacción que aporte valor (hover enriquecido o filtro); prueba también zoom y selección donde sean pertinentes.
5. Dos hallazgos con valores comprobables y una limitación.
6. Justificación de la interacción y una situación donde sería preferible la figura estática.

La celda siguiente es un **punto de partida ejecutable**, no una solución completa. Cambia la pregunta y las decisiones; no presentes el resultado sin interpretarlo.

In [ ]:
actividad = base.groupby('tipo').size().nlargest(8).rename('registros').reset_index()
fig, ax = plt.subplots(figsize=(11,5))
sns.barplot(data=actividad, x='registros', y='tipo', color='#287C8E', errorbar=None, ax=ax)
ax.set(xlabel='Número de registros', ylabel='Tipo', title='Ocho tipos con más registros · fuente 2026')
plt.tight_layout(); plt.show()
fig_actividad = px.bar(actividad.sort_values('registros'), x='registros', y='tipo', orientation='h',
    title='Ocho tipos con más registros · fuente 2026', labels={'registros':'Número de registros','tipo':'Tipo de incidente'},
    color_discrete_sequence=['#287C8E'])
fig_actividad.update_layout(height=500, margin=dict(l=260))
mostrar(fig_actividad)

### Respuesta del estudiante
**Audiencia y pregunta:** …  
**Unidad y cobertura:** …  
**Hallazgo 1 y evidencia:** …  
**Hallazgo 2 y evidencia:** …  
**Limitación:** …  
**Valor de la interacción:** …

### Cierre y verificación
- [ ] Puedo explicar zoom, hover, selección y filtrado dinámico con ejemplos distintos.
- [ ] Mis figuras Seaborn/Plotly usan iguales datos y unidades.
- [ ] Distingo conteos de registros, incidentes únicos y tasas poblacionales.
- [ ] Reconozco la cobertura desigual de las fuentes de 2025 y 2026.
- [ ] Justifico mis controles desde la audiencia y la pregunta.

**Resolución de problemas:** si faltan archivos, vuelve a la sección 2.1; si hay varias copias, conserva la versión elegida con el nombre exacto. Si un gráfico no aparece en Colab, ejecuta la celda de importación y luego su celda. Los controles se usan dentro del gráfico; no necesitan volver a ejecutar Python.

## Referencias y correspondencia con la guía
Guía Docente de Visualización I, sesión 23, página 7: biblioteca e integración web/Jupyter (sección 1), cuatro formas de interacción (sección 3), conversión de Seaborn y diferencias de diseño (sección 4), evaluación del valor comunicativo y conexión con Power BI (sección 8).

Documentación oficial consultada:
- [Renderers y Colab](https://plotly.com/python/renderers/)
- [Hover](https://plotly.com/python/hover-text-and-formatting/)
- [Selecciones](https://plotly.com/python/selections/)
- [Menús desplegables](https://plotly.com/python/dropdowns/)

Los resultados se calculan con tus archivos al ejecutar el Notebook. No se incluyen conclusiones numéricas inventadas.